================================================================================
模块 06：现代投资组合理论与仓位风控算法 · 第 4/5 课
第 4 课：风险价值 (VaR)、条件风险价值 (CVaR) 与动态回撤熔断控制 (VaR, CVaR & Drawdown Sentinel)
================================================================================

【课程定位与第一性原理】
在现代金融机构监管与量化资管资本充足率评价中，波动率（$\sigma_p$）已不足以刻画非对称的左侧极端尾部损失。
巴塞尔协议（Basel Accords）确立了两大核心风险度量标准：风险价值（VaR）与条件风险价值（CVaR / Expected Shortfall）。
投资委员会在审视量化产品风控底线时，必须穿透高斯正态假设在极端黑天鹅面前的虚伪粉饰，
建立一致性风险度量与基于历史最高水位线（High-Water Mark, HWM）的动态阶梯式熔断断路器。

- **[VaR 的黑天鹅盲区与次可加性失效]**：VaR 仅度量分位数阈值，对击穿阈值后的损失深浅完全失明，且在非线性资产中不满足次可加性公理。
- **[CVaR 的一致性风险度量优越性]**：CVaR 积分计算击穿 VaR 后的条件期望损失，严格满足凸性与次可加性公理，已成为巴塞尔协议 III 的绝对基准。
- **[动态高水位哨兵与熔断断路器]**：逐日追踪历史最高净值峰值与回撤深度，实施“-5% 一级预警降半仓、-10% 二级硬熔断冷冻”的严格因果风控纪律。

【核心概念与就近图文排版】

1. 正态假设 vs 真实肥尾分布下的 VaR 与 CVaR 积分：
观察下图关于正态分布与肥尾分布在尾部损失积分（CVaR）上的巨大差距：

<div class="diagram-card">
  <div class="diagram-header">
    <span class="diagram-title">图 6.4.1：正态分布 vs 真实厚尾分布：VaR 阈值盲区与 CVaR 条件期望尾部损失</span>
    <span class="diagram-tag">VAR-VS-CVAR-TAILS</span>
  </div>
  <div class="diagram-svg-container">
    <!-- SVG 锚点: assets/m06_l04_var_vs_cvar_tail_distribution.svg -->
  </div>
</div>

- **[参数法 VaR 的正态粉饰]**：假定收益率服从正态分布，$\text{VaR}_{95\%} = 1.645 \sigma - \mu$。在金融肥尾危机面前严重低估极端损失。
- **[历史模拟法非参数 CVaR]**：直接从真实经验分布中提取 $\alpha$ 尾部样本均值：$\text{CVaR} = -\mathbb{E}[R \mid R \le -\text{VaR}]$，保留真实黑天鹅冲击。
- **[肥尾倒挂倍数]**：在 Student-t 厚尾市场中，真实历史 CVaR 往往是参数法正态 VaR 的 2~3 倍以上。

2. 历史最高水位线 (HWM) 与水下回撤刻画：
下图展示了累计财富曲线与历史最高峰值（HWM）的阶梯推进与水下回撤（Underwater）深度图谱：

<div class="diagram-card">
  <div class="diagram-header">
    <span class="diagram-title">图 6.4.2：动态历史最高水位线 (HWM) 与水下回撤 (Underwater Chart) 深度刻画</span>
    <span class="diagram-tag">HIGH-WATER-MARK</span>
  </div>
  <div class="diagram-svg-container">
    <!-- SVG 锚点: assets/m06_l04_drawdown_high_water_mark.svg -->
  </div>
</div>

- **[最高水位线 HWM]**：$H_t = \max_{s \le t} W_s$，单调递增阶梯式上升，记录策略所达到的历史最巅峰财富。
- **[动态回撤率 DD]**：$\text{DD}_t = \frac{W_t - H_t}{H_t}$，恒为非正值（$\le 0$），直观反映投资者的实际被套深度。
- **[水下时间 (Time Underwater)]**：从发生回撤到重新刷新 HWM 创出新高所耗费的时间跨度，度量 LP 资金的心理耐受极限。

3. 阶梯式回撤风控哨兵状态转移模型：
下图展示了多策略量化系统内部基于回撤阈值的自适应降仓与硬熔断状态机：

<div class="diagram-card">
  <div class="diagram-header">
    <span class="diagram-title">图 6.4.3：阶梯式动态回撤风控哨兵状态转移模型 (Drawdown Sentinel State Machine)</span>
    <span class="diagram-tag">SENTINEL-CIRCUITS</span>
  </div>
  <div class="diagram-svg-container">
    <!-- SVG 锚点: assets/m06_l04_circuit_breaker_sentinel_states.svg -->
  </div>
</div>

- **[正常运行态 (Normal)]**：$\text{DD} > -5\%$，仓位乘数 $M = 1.0$，策略全功率运行。
- **[一级降仓预警态 (Warning)]**：$-10\% < \text{DD} \le -5\%$，仓位乘数压缩至 $M = 0.5$，主动削减组合方差以平抑巨震。
- **[二级硬熔断冷冻态 (Halted)]**：$\text{DD} \le -10\%$，仓位强制清零 $M = 0.0$，策略全部平仓置换为无风险现金，进入人工投决会复盘审查。

<div class="callout callout-note">
  <div class="callout-title">📌 投决会核心心智模型：风控前置与资本保全的绝对性</div>
  <div class="callout-body">
    在对冲基金行业中，收益率决定了基金经理能飞多高，而回撤控制决定了基金能活多久。当净值回撤触及熔断红线时，任何关于“基本面被错杀即将反弹”的侥幸辩解都是非法的，算法必须冷酷执行硬熔断。
  </div>
</div>

<div class="callout callout-warning">
  <div class="callout-title">⚠️ 投决会风控警示与认知陷阱</div>
  <div class="callout-body">
    风控决策必须严格实施因果时序滞后（$T-1$ 日回撤决定 $T$ 日仓位乘数），严禁用当日包含盘中未结价格的高水位直接作为当日收盘风控判决，杜绝日内前视偏差作弊。
  </div>
</div>

【数学与风控模型】
$$\text{VaR}_{\alpha} = -Q_{\alpha}(R), \quad \text{CVaR}_{\alpha} = -\frac{1}{\alpha} \int_0^{\alpha} Q_p(R) dp$$
$$\text{Parametric}: \quad \text{VaR}_{\alpha} = z_{\alpha} \sigma - \mu, \quad \text{CVaR}_{\alpha} = \sigma \frac{\phi(z_{\alpha})}{\alpha} - \mu$$
$$\text{HWM}_t = \max_{s \le t} W_s, \quad \text{DD}_t = \frac{W_t - \text{HWM}_t}{\text{HWM}_t}$$
$$M_t = \begin{cases} 
1.0, & \text{if } \text{DD}_{t-1} > \text{Warning\_DD} \\
0.5, & \text{if } \text{Circuit\_Breaker\_DD} < \text{DD}_{t-1} \le \text{Warning\_DD} \\
0.0, & \text{if } \text{DD}_{t-1} \le \text{Circuit\_Breaker\_DD}
\end{cases}$$
定义域：$\alpha = 0.05 \text{ (95% 置信度)}, \quad \text{Warning\_DD} = -0.05, \quad \text{Circuit\_Breaker\_DD} = -0.10$。

【思考题】
Q1: 风险价值（VaR）的“黑天鹅盲区”与次可加性（Sub-additivity）公理失效：为什么阿茨纳（Artzner et al. 1999）将 VaR 斥为“非一致性风险度量指标”？请构造一个期权空头组合或极端肥尾组合，说明为什么在特定情况下两个低风险子组合合并后，整体组合的 VaR 竟然大于子组合 VaR 之和？为什么巴塞尔协议 III 强制采用 CVaR（Expected Shortfall）替代 VaR？
Q2: 参数法正态 VaR 在 2008 / 2020 危机中的系统性穿仓：正态分布假设下，收益率偏离均值 6 个标准差（$6\sigma$）的发生概率为数亿年一次。但在真实金融危机中，为何连续多日出现所谓的 $6\sigma \sim 8\sigma$ 事件？肥尾柯西/帕累托分布如何彻底击穿参数法 VaR 的防线？
Q3: 动态历史最高水位线（HWM）阶梯式风控哨兵的因果性契约：在量化私募实盘风控中，为什么要实施“回撤超 -5% 降半仓，回撤超 -10% 硬熔断全现金冷冻”的阶梯式断路器？在算法架构上，为什么该风控决策必须严格滞后 1 期（Shift-1）执行？

【评分与通过规则】
- 代码 4 分：参数法正态 VaR/CVaR 计算 (1分) + 经验分布历史模拟法计算 (1.5分) + 动态高水位熔断哨兵模拟引擎 (1.5分)
- Q1~Q3 深度思考题各 2 分（共 6 分）：从次可加性公理、肥尾极端损失积分与阶梯式断路器因果隔离展开
- 通过线：8/10 分。一票否决：VaR 以负数损失表达、熔断器未实施 T-1 滞后执行


In [ ]:
from typing import Tuple, Dict, Any, List
import numpy as np
import pandas as pd
from scipy import stats


def calculate_parametric_var_cvar(
    returns: np.ndarray,
    confidence: float = 0.95,
) -> Tuple[float, float]:
    """
    基于正态分布假定的参数法单期 VaR 与 CVaR (以正数表示损失率)：
        alpha = 1 - confidence
        VaR_param = z_alpha * sigma - mu
        CVaR_param = sigma * (phi(z_alpha) / alpha) - mu

    参数:
        returns: 资产历史收益率数组 (一维)
        confidence: 置信水平 (默认 0.95)

    返回:
        Tuple[float, float]: (参数法 VaR, 参数法 CVaR)
    """
    # TODO: 实现参数法正态 VaR 与 CVaR
    pass


def calculate_historical_var_cvar(
    returns: np.ndarray,
    confidence: float = 0.95,
) -> Tuple[float, float]:
    """
    基于经验分布的历史模拟法单期 VaR 与 CVaR (以正数表示损失率)：
        alpha = 1 - confidence
        cutoff = percentile(returns, alpha * 100)
        VaR_hist = -cutoff
        CVaR_hist = -mean(returns[returns <= cutoff])

    参数:
        returns: 资产历史收益率数组 (一维)
        confidence: 置信水平 (默认 0.95)

    返回:
        Tuple[float, float]: (历史模拟法 VaR, 历史模拟法 CVaR)
    """
    # TODO: 实现非参数历史模拟法 VaR 与 CVaR
    pass


def simulate_dynamic_drawdown_sentinel(
    raw_returns: pd.Series,
    warning_dd: float = -0.05,
    cut_ratio: float = 0.50,
    circuit_breaker_dd: float = -0.10,
) -> Dict[str, Any]:
    """
    基于历史高水位线 (HWM) 的动态阶梯式回撤风控哨兵：
    逐日滚动跟踪策略累计财富 W_t 与历史最高水位线 HWM_t = max_{s <= t} W_s：
    回撤率: DD_t = (W_t - HWM_t) / HWM_t

    风控执行规则 (决策必须滞后 1 期执行以防止前视偏差):
    - 若上一日 DD > warning_dd: 仓位乘数 pos = 1.0 (正常)
    - 若上一日 circuit_breaker_dd < DD <= warning_dd: 仓位乘数 pos = cut_ratio (0.5 降仓预警)
    - 若上一日 DD <= circuit_breaker_dd: 仓位乘数 pos = 0.0 (硬熔断平仓冷冻)

    参数:
        raw_returns: 未受风控约束的原始资产单日收益率 Series
        warning_dd: 一级降仓预警回撤阈值 (负数, 默认 -0.05)
        cut_ratio: 一级预警状态下的仓位压缩乘数 (默认 0.50)
        circuit_breaker_dd: 二级硬熔断回撤阈值 (负数, 默认 -0.10)

    返回字典包含:
        - "raw_max_drawdown": 原始策略最大回撤 (负值)
        - "controlled_max_drawdown": 受控策略最大回撤 (负值)
        - "controlled_returns": 受控策略每日收益率 Series
        - "drawdown_reduction_pct": 最大回撤削减幅度 ((ctrl_dd - raw_dd) / abs(raw_dd))
        - "circuit_breaker_tripped": 是否触发了二级硬熔断 (bool)
    """
    # TODO: 实现动态回撤风控哨兵模拟引擎
    pass


if __name__ == "__main__":
    print("=" * 70)
    print("运行 Lesson 04: VaR / CVaR 与动态回撤熔断单元测试")
    print("=" * 70)

    # 1. 验证简单离散样本下的历史模拟法 VaR 与 CVaR
    rets_simple = np.array([-0.10, -0.06, -0.05, -0.02, 0.01, 0.02, 0.03, 0.04, 0.05, 0.08])
    h_var90, h_cvar90 = calculate_historical_var_cvar(rets_simple, confidence=0.90)
    print(f"简单样本 90% 历史 VaR: {h_var90*100:.2f}%, 历史 CVaR: {h_cvar90*100:.2f}%")
    assert h_var90 > 0.0, "VaR 以正损失率表达"
    assert h_cvar90 >= h_var90, "根据定义, 极端条件期望 CVaR 必须大于等于 VaR"

    # 2. 对比厚尾分布 (Student-t, df=3) 下参数法与历史模拟法的巨大反差
    np.random.seed(42)
    # 模拟 2000 个带厚尾特性的金融资产日收益率
    t_rets = stats.t.rvs(df=3, loc=0.0005, scale=0.015, size=2000)

    p_var, p_cvar = calculate_parametric_var_cvar(t_rets, confidence=0.95)
    h_var, h_cvar = calculate_historical_var_cvar(t_rets, confidence=0.95)

    print(f"\n厚尾数据 95% 参数法 VaR: {p_var*100:.2f}%, 参数法 CVaR: {p_cvar*100:.2f}%")
    print(f"厚尾数据 95% 历史法 VaR: {h_var*100:.2f}%, 历史法 CVaR: {h_cvar*100:.2f}%")

    assert p_cvar >= p_var, "参数法 CVaR 必须大于等于参数法 VaR"
    assert h_cvar >= h_var, "历史模拟法 CVaR 必须大于等于历史法 VaR"
    # 在极度厚尾数据下，真实历史 CVaR 必须显著高于正态分布假定的参数法 CVaR
    assert h_cvar > p_cvar, f"厚尾极端损失预期应显著击穿正态假设: 历史 CVaR ({h_cvar}) > 参数 CVaR ({p_cvar})"

    # 3. 验证动态高水位回撤控制与阶梯式熔断器
    n_days = 252
    raw_rets_arr = np.random.normal(0.001, 0.015, n_days)
    # 在第 100~150 天注入持续深幅崩盘行情
    raw_rets_arr[100:150] = np.random.normal(-0.008, 0.02, 50)
    raw_series = pd.Series(raw_rets_arr)

    sentinel_res = simulate_dynamic_drawdown_sentinel(
        raw_series,
        warning_dd=-0.05,
        cut_ratio=0.50,
        circuit_breaker_dd=-0.10,
    )

    print(f"\n未受控原始策略最大回撤: {sentinel_res['raw_max_drawdown']*100:.2f}%")
    print(f"受控策略最大回撤: {sentinel_res['controlled_max_drawdown']*100:.2f}%")
    print(f"回撤削减幅度: {sentinel_res['drawdown_reduction_pct']*100:.2f}%")
    print(f"是否触发硬熔断: {sentinel_res['circuit_breaker_tripped']}")

    assert sentinel_res["raw_max_drawdown"] < -0.30, "原始策略在暴跌注入期间应发生深幅回撤 (< -30%)"
    # 动态熔断器应将最大回撤牢牢锁定在熔断线附临近区间 (不发生大于 -15% 的深度穿仓)
    assert sentinel_res["controlled_max_drawdown"] > -0.15, f"受控策略最大回撤应受控在 -15% 以内, 实际: {sentinel_res['controlled_max_drawdown']*100:.2f}%"
    assert sentinel_res["drawdown_reduction_pct"] > 0.50, "回撤削减幅度应超过 50%"
    assert sentinel_res["circuit_breaker_tripped"] is True, "暴跌行情应成功触发二级硬熔断保护"

    print("Lesson 04 风险计量与动态回撤风控单元测试全部通过！")
